In [1]:
import pandas as pd
import torch

In [2]:
df_emb = pd.read_csv('molformer-emb.csv')
df_emb

,0,1,2,3,4,5,6,7,8,9,...,759,760,761,762,763,764,765,766,767,MP
0,0.232754,0.039662,0.603345,0.434351,-0.235619,-0.008117,-0.240256,-0.327148,-0.622383,-0.242045,...,0.511787,0.017956,-0.102726,0.593849,-0.322905,-2.748505,0.264649,-0.493542,0.170734,355.15
1,-0.035469,0.244099,0.824150,0.871007,0.381988,0.018918,-0.171494,-0.681534,0.314292,0.073357,...,-0.444052,-0.211882,-0.301885,-0.089473,-0.258444,-2.224507,0.016750,-0.915583,0.149851,373.65
2,-0.014357,0.257718,0.659238,0.541999,0.342051,0.048318,0.188520,-0.241498,-0.025705,-0.207805,...,-0.124264,0.174638,-0.048913,-0.015672,-0.385274,-2.904102,0.212661,-0.588502,-0.315535,373.75
3,0.392396,0.213285,0.820162,0.444167,-0.283028,-0.213827,0.098154,-0.381828,0.016981,-0.291995,...,-0.253576,-0.132808,0.243490,0.575032,0.081913,-2.777802,0.065183,-0.286023,-0.647094,373.70
4,0.630157,0.129021,0.930085,0.348113,-0.079924,-0.057398,0.470295,0.097515,-0.180795,-0.171304,...,0.135372,-0.524551,0.513307,-0.236967,-0.219924,-3.077367,0.617281,-0.337038,-0.176350,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0.536852,0.122638,0.304402,-0.319562,-0.675866,0.010863,-0.525479,-0.008692,-0.520580,-0.474743,...,-0.322050,-0.702077,0.462894,0.641812,-0.209804,-2.338952,0.601882,-0.391491,-0.380140,395.65
117748,-0.091166,0.245632,-0.179062,0.299272,-0.800757,-0.122798,-0.938942,-0.098464,0.014622,-0.054302,...,-0.299069,0.062411,0.707462,0.116971,-0.379527,-2.399415,0.433567,-0.402556,-0.216447,396.15
117749,0.463926,0.466987,0.087009,0.034810,-1.054933,-0.277966,-0.886445,-0.060067,-0.231398,0.083403,...,0.006923,-0.156032,0.417531,0.423048,-0.546653,-2.341095,0.170920,-0.561870,-0.619896,327.65
117750,0.502144,0.065869,0.162445,-0.160113,-1.070973,-0.249068,-0.573376,-0.082146,0.197232,-0.234873,...,-0.269982,-0.340257,0.310560,0.339327,-0.547660,-2.511800,0.631499,-0.330420,-0.576203,488.15


In [34]:
from sklearn.model_selection import train_test_split
X_emb_train, X_emb_test, Y_train, Y_test = train_test_split(df_emb.iloc[:, :-1], df_emb.iloc[:, -1], test_size=0.2, random_state=42)

In [3]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [36]:
%%time
model = xgb.XGBRegressor(n_estimators=1000, n_jobs=-1, learning_rate=0.05, max_depth=7)

model.fit(X_emb_train, Y_train)

Y_pred = model.predict(X_emb_test)

mse = mean_squared_error(Y_test, Y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print(f"MSE: {mse}, RMSE: {rmse}, MAE: {mae}, R²: {r2}")

MSE: 1556.941013299498, RMSE: 39.458091860852804, MAE: 30.251991138803454, R²: 0.6937748464197527
CPU times: user 18h 22min 40s, sys: 25min 14s, total: 18h 47min 54s
Wall time: 58min 38s


In [37]:
print(f"MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.2f}")

MSE: 1556.94, RMSE: 39.46, MAE: 30.25, R²: 0.69


In [38]:
import timeit

# Warm-up (ignore first run)
_ = model.predict(X_emb_test[:1])

# Configuration
num_test_samples = X_emb_test.shape[0]  # Number of samples in test set
num_runs = 100  # Adjust based on how precise you need the measurement

# Measure total time for 'num_runs' full-batch predictions
total_time = timeit.timeit(lambda: model.predict(X_emb_test), number=num_runs)

# Calculate average time per run (for the full batch)
avg_time_per_batch = total_time / num_runs

# Calculate time per individual sample
avg_time_per_sample = avg_time_per_batch / num_test_samples

print(f"Avg. time per full batch ({num_test_samples} samples): {avg_time_per_batch * 1000:.2f} ms")
print(f"Avg. time per sample: {avg_time_per_sample * 1000:.6f} ms")
print(f"Throughput: {num_test_samples / avg_time_per_batch:.2f} samples/sec")

Avg. time per full batch (23551 samples): 340.85 ms
Avg. time per sample: 0.014473 ms
Throughput: 69095.04 samples/sec


In [2]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

/opt/anaconda/envs/torch_gcnn/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


In [3]:
import torch
torch.cuda.empty_cache()

In [4]:
df = pd.read_csv('df.csv')
df.isna().sum()

MP        0
SMILES    0
dtype: int64

In [10]:
df = pd.read_csv('df.csv')
df

,MP,SMILES
0,355.15,O=[N+]([O-])c1c(O)cccc1O
1,373.65,Cc1c([N+](=O)[O-])cc([N+](=O)[O-])c(N(C)[N+](=...
2,373.75,O=c1ccc([N+](=O)[O-])cn1Cc1ccccc1
3,373.70,N#CC(Cc1ccccc1)(Cc1ccco1)c1ccc([N+](=O)[O-])cc1
4,374.25,CCOC(=O)c1cc2ccccc2cc1-c1ccc([N+](=O)[O-])cc1
...,...,...
117747,395.65,BrC1=CC2=C(C=C1)N=C(O2)C1=CC=CC=C1N(=O)=O
117748,396.15,C[C@@H]1CC(=O)C[C@@H](N1)C1=CC=C(C=C1)N(=O)=O
117749,327.65,O=N(=O)C1=C(OCC#N)C=CC=C1
117750,488.15,O=N(=O)C1=CC(=C(NN=C\C=C\C2=CC=CO2)C=C1)N(=O)=O


In [11]:
df = df.rename(columns = {'MP' : 'label', 'SMILES' : 'smiles'})
df

,label,smiles
0,355.15,O=[N+]([O-])c1c(O)cccc1O
1,373.65,Cc1c([N+](=O)[O-])cc([N+](=O)[O-])c(N(C)[N+](=...
2,373.75,O=c1ccc([N+](=O)[O-])cn1Cc1ccccc1
3,373.70,N#CC(Cc1ccccc1)(Cc1ccco1)c1ccc([N+](=O)[O-])cc1
4,374.25,CCOC(=O)c1cc2ccccc2cc1-c1ccc([N+](=O)[O-])cc1
...,...,...
117747,395.65,BrC1=CC2=C(C=C1)N=C(O2)C1=CC=CC=C1N(=O)=O
117748,396.15,C[C@@H]1CC(=O)C[C@@H](N1)C1=CC=C(C=C1)N(=O)=O
117749,327.65,O=N(=O)C1=C(OCC#N)C=CC=C1
117750,488.15,O=N(=O)C1=CC(=C(NN=C\C=C\C2=CC=CO2)C=C1)N(=O)=O


In [ ]:
%%time

import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

X = df_emb.iloc[:, :-1]
y = df_emb.iloc[:, -1]

n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

maes = []
rmsses = []

for train_idx, val_idx in kf.split(X):

    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(
        n_estimators=1000,
        n_jobs=-1,
        learning_rate=0.05,
        max_depth=7
    )
    model.fit(X_train, y_train)
 
    y_pred = model.predict(X_val)

    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_val, y_pred)
    
    rmsses.append(rmse)
    maes.append(mae)

mean_mae = np.mean(maes)
std_mae = np.std(maes)
mean_rmse = np.mean(rmsses)
std_rmse = np.std(rmsses)

print(f'MAE: {mean_mae:.4f} ± {std_mae:.4f}')
print(f'RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}')

In [5]:
print(f'MAE: {mean_mae:.4f} ± {std_mae:.4f}')
print(f'RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}')

MAE: 30.4799 ± 0.2355
RMSE: 39.8119 ± 0.3440


In [6]:
print(f'MAE: {mean_mae:.4f} ± {std_mae:.4f}')
print(f'RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}')

MAE: 30.4799 ± 0.2355
RMSE: 39.8119 ± 0.3440
